# לפלס ופואסון עם תנאי דיריכלה — מלבן ודיסקה

מחברת CPU ללא אימון. פותרים $-\Delta u=f$ בתחום ו־$u=g$ על השפה. אותה תשתית HilbertSpace משמשת לנורמות ולאינטגרציה; הסולבר משתמש באלמנטים סופיים, ולא במכפילי פורייה.

## הכנת הסביבה
ב־Colab העלו את חבילת הקוד המצורפת כאשר תא ההכנה מבקש אותה. אין צורך בנתוני פייז 6.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
PROJECT_ROOT = Path(os.environ.get("SPNO_PROJECT_ROOT", "/content/spno-colab" if IN_COLAB else str(Path.cwd())))
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    if not (PROJECT_ROOT / "src/spno/workflow.py").exists():
        # Upload the supplied spno-colab-source.zip once per fresh runtime.
        import zipfile
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload the single spno-colab-source.zip bundle")
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                if not (PROJECT_ROOT / member.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive member")
            archive.extractall(PROJECT_ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT) + "[dirichlet,notebooks]"])
if not (PROJECT_ROOT / "src/spno/workflow.py").is_file():
    raise FileNotFoundError("Set PROJECT_ROOT to the updated spno source directory")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
OPTIONS = json.loads(os.environ.get("SPNO_OPTIONS", "{}"))
from IPython.display import display, HTML, Image


## רשתות ותנאי שפה
במלבן מחלקים רשת סדורה למשולשים. בדיסקה משתמשים בטבעות ובמרכז יחיד; השפה היא מצולע שמתקרב למעגל בעידון. ערכי השפה נקבעים בצמתים, ומורחבים ליניארית לאורך הצלעות.

בודקים לפלס עם $u=x+2y$, ופואסון עם $u=\sin(x)\cos(y)$ ו־$f=2u$. תנאי השפה מתקבלים מהפתרונות הידועים. המערכת נפתרת ב־float64.

In [ ]:
from spno.dirichlet import demo
OUTPUT_ROOT = Path(os.environ.get("SPNO_OUTPUT_ROOT", "/content/drive/MyDrive/spno/workflow" if IN_COLAB else str(PROJECT_ROOT / "results/colab-workflow")))
report, figure = demo(refinements=OPTIONS.get("refinements", [4, 8, 16]), output=OUTPUT_ROOT / "dirichlet")
display(figure)
print(json.dumps(report, indent=2))


## ניסוי משלכם
החליפו את $f$ ואת ערכי $g$ בצמתים. כאן פותרים פואסון בדיסקה עם פתרון ייחוס $u=1-r^2$; שגיאת הגאומטריה והאינטרפולציה קטנה עם עידון השפה.

In [ ]:
import torch
from spno.dirichlet import disk_mesh, solve_dirichlet
mesh = disk_mesh(rings=8, angles=64)
x, y = mesh.points.T
exact = 1 - x.square() - y.square()
solution = solve_dirichlet(mesh, torch.full_like(x, 4.0), exact)
print("Weighted nodal L2 error:", float(mesh.norm(solution.values - exact)))
print("Linear-system residual:", solution.relative_residual)
